In [1]:
import sys
import os

sibling_dir = os.path.abspath('../01_agentic_rag')

if sibling_dir not in sys.path:
    sys.path.append(sibling_dir)

In [2]:
from dotenv import load_dotenv
load_dotenv()

# configure the OpenAI client with your API key
from openai import OpenAI

openai_client = OpenAI()

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [5]:
query = "How do I run Kafka?"

query_vector = model.encode(query)
results = vs_index.search(query_vector, num_results=5, filter_dict={"course": "llm-zoomcamp"})
results

[{'id': '04440cab11',
  'course': 'llm-zoomcamp',
  'section': 'Module 3: Orchestration',
  'question': 'Kestra AI Copilot replies "I can only assist with creating Kestra flows" — how do I fix it?',
  'answer': 'This message means the AI Copilot didn\'t get a valid Gemini API key, so it falls back to a canned refusal. In Kestra\'s Open Source edition the Copilot only supports Gemini, and it reads the **plain** `GEMINI_API_KEY` variable (not the base64-encoded `SECRET_GEMINI_API_KEY` that the flows use).\n\nMake sure you exported the plain key before starting Kestra, then restart it:\n\n```bash\nexport GEMINI_API_KEY="your-gemini-api-key-here"\ndocker compose up -d\n```\n\nIf it still fails, the key is usually missing, mistyped, or rate-limited:\n\n- Confirm the variable is actually set in the shell you ran `docker compose up` from (`echo $GEMINI_API_KEY`).\n- Generate a fresh key in [Google AI Studio](https://aistudio.google.com/app/apikey).\n- If you\'ve been running the agent/multi-a

In [6]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [8]:
# we subclass RAGBase and override search to encode the query before searching
from rag_helper import RAGBase

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )


In [9]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [10]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join. If you want a certificate, make sure you submit your project while submissions are still open.'

In [11]:
vs_index.close()